# RAG vs Prompt-Only vs Hybrid — Three-Way Comparative Evaluation

**Purpose:** Three-way experimental comparison for the Bachelor thesis.  
**Protocol:** Same questions → same Gemini model → same temperature → blind-gradable outputs.  

| | RAG System | Prompt-Only System | Hybrid System |
|---|---|---|---|
| **Engine** | `rag_engine.py` | `prompt_only_engine.py` | `hybrid_engine.py` |
| **Knowledge source** | ChromaDB only | System prompt only | RAG + prompt (routed) |
| **Routing** | Always retrieval | Never retrieval | Case A/B/C |
| **LLM model** | Same Gemini | Same Gemini | Same Gemini |
| **Temperature** | 0.15 | 0.15 | 0.15 |
| **Max tokens** | 4096 | 4096 | 4096 |

---

## 0. Setup & Initialization

In [ ]:
import sys
import os
import time
import json
from datetime import datetime, timezone
from pathlib import Path

print(f"Python  : {sys.version}")
print(f"CWD     : {os.getcwd()}")
print()

In [ ]:
import config
print(f"Gemini model : {config.CHAT_MODEL_ID}")
print(f"Temperature  : 0.15 (hardcoded in all engines)")
print(f"Max tokens   : 4096 (hardcoded in all engines)")
print(f"Good thresh  : {config.SIMILARITY_GOOD_THRESHOLD}")
print(f"Fallback thr : {config.SIMILARITY_FALLBACK_THRESHOLD}")
print(f"Hybrid cap   : {config.HYBRID_CASE_B_MAX_CONFIDENCE}")

In [ ]:
# ── Initialize all three engines ─────────────────────────────
from rag_engine import TunisianMathRAG
from prompt_only_engine import TunisianMathPromptOnly
from hybrid_engine import TunisianMathHybrid

t0 = time.monotonic()
rag_engine = TunisianMathRAG()
rag_init = time.monotonic() - t0
print(f"RAG engine ready in {rag_init:.1f}s | {rag_engine.chunk_count:,} chunks")

t0 = time.monotonic()
po_engine = TunisianMathPromptOnly()
po_init = time.monotonic() - t0
print(f"Prompt-only engine ready in {po_init:.1f}s")

t0 = time.monotonic()
hybrid_engine = TunisianMathHybrid()
hybrid_init = time.monotonic() - t0
print(f"Hybrid engine ready in {hybrid_init:.1f}s | {hybrid_engine.chunk_count:,} chunks")

print(f"\nInit times: RAG={rag_init:.1f}s | PO={po_init:.1f}s | Hybrid={hybrid_init:.1f}s")

---
## 1. Evaluation Query Bank

**10 queries** covering the main Bac Math chapters.  
Each query will be sent to ALL THREE systems with the SAME mode.

In [ ]:
EVAL_QUERIES = [
    # ── CORRECTION MODE ──
    {
        "id": "Q01",
        "question": "Résoudre dans C l'équation z² - 4z + 8 = 0. Écrire les solutions sous forme trigonométrique.",
        "mode": "correction",
        "chapter": "Nombres complexes",
        "difficulty": "standard",
    },
    {
        "id": "Q02",
        "question": "Soit (Un) définie par U0 = 2 et Un+1 = (1/2)Un + 1. Montrer que (Un) converge et calculer sa limite.",
        "mode": "correction",
        "chapter": "Suites numériques",
        "difficulty": "standard",
    },
    {
        "id": "Q03",
        "question": "Calculer l'intégrale I = ∫₀¹ x·eˣ dx en utilisant une intégration par parties.",
        "mode": "correction",
        "chapter": "Intégration",
        "difficulty": "standard",
    },
    {
        "id": "Q04",
        "question": "Déterminer le PGCD de 1071 et 1029 par l'algorithme d'Euclide. En déduire une relation de Bézout.",
        "mode": "correction",
        "chapter": "Arithmétique",
        "difficulty": "standard",
    },
    {
        "id": "Q05",
        "question": "Résoudre l'équation différentielle y' - 3y = 6 avec la condition y(0) = 1.",
        "mode": "correction",
        "chapter": "Équations différentielles",
        "difficulty": "standard",
    },
    # ── COACHING MODE ──
    {
        "id": "Q06",
        "question": "Comment étudier la continuité et la dérivabilité d'une fonction en un point ? Donne-moi la méthode avec un exemple.",
        "mode": "coaching",
        "chapter": "Fonctions numériques",
        "difficulty": "conceptuel",
    },
    {
        "id": "Q07",
        "question": "Explique-moi comment déterminer une équation cartésienne d'un plan dans l'espace à partir de 3 points.",
        "mode": "coaching",
        "chapter": "Géométrie dans l'espace",
        "difficulty": "conceptuel",
    },
    {
        "id": "Q08",
        "question": "Dans un sac il y a 4 boules rouges et 6 bleues. On tire 3 boules sans remise. Quelle est la probabilité d'avoir exactement 2 rouges ?",
        "mode": "coaching",
        "chapter": "Probabilités",
        "difficulty": "standard",
    },
    {
        "id": "Q09",
        "question": "Résoudre dans R : ln(x+3) + ln(x-1) = ln(5). Explique chaque étape.",
        "mode": "coaching",
        "chapter": "Logarithme & Exponentielle",
        "difficulty": "standard",
    },
    {
        "id": "Q10",
        "question": "Soit f(x) = x³ - 3x + 1. Faire l'étude complète de f (domaine, limites, dérivée, tableau de variation, courbe).",
        "mode": "coaching",
        "chapter": "Études de fonctions",
        "difficulty": "complet",
    },
]

print(f"Evaluation bank: {len(EVAL_QUERIES)} queries")
print(f"  Correction mode: {sum(1 for q in EVAL_QUERIES if q['mode'] == 'correction')}")
print(f"  Coaching mode  : {sum(1 for q in EVAL_QUERIES if q['mode'] == 'coaching')}")
print(f"\nChapters covered:")
for q in EVAL_QUERIES:
    print(f"  {q['id']} | {q['chapter']:<30s} | {q['mode']:<11s} | {q['difficulty']}")

---
## 2. Run All Three Systems

For each query: RAG → Prompt-Only → Hybrid.  
**Rate-limit protection:** delays between API calls.  
**Estimated API cost:** ~30 Gemini calls (10 queries × 3 systems)  
**Estimated time:** ~15-20 min

In [ ]:
INTER_QUERY_DELAY = 5   # seconds between queries
INTER_SYSTEM_DELAY = 2  # seconds between systems within same query

comparison_results = []

for i, q in enumerate(EVAL_QUERIES, 1):
    print(f"\n{'='*80}")
    print(f"  QUERY {q['id']} ({i}/{len(EVAL_QUERIES)}): {q['chapter']}")
    print(f"  Mode: {q['mode']} | Difficulty: {q['difficulty']}")
    print(f"  Q: {q['question']}")
    print(f"{'='*80}")

    # ── RAG System ──
    print(f"\n  [RAG] Running...")
    rag_result = rag_engine.query(q["question"], mode=q["mode"])
    print(f"  [RAG] Done: case={rag_result.retrieval_case} "
          f"conf={rag_result.confidence} "
          f"docs={len(rag_result.selected_docs)} "
          f"total={rag_result.total_time:.2f}s")
    if rag_result.error:
        print(f"  [RAG] ERROR: {rag_result.error}")

    time.sleep(INTER_SYSTEM_DELAY)

    # ── Prompt-Only System ──
    print(f"  [PROMPT-ONLY] Running...")
    po_result = po_engine.query(q["question"], mode=q["mode"])
    print(f"  [PROMPT-ONLY] Done: "
          f"conf={po_result.confidence} "
          f"total={po_result.total_time:.2f}s")
    if po_result.error:
        print(f"  [PROMPT-ONLY] ERROR: {po_result.error}")

    time.sleep(INTER_SYSTEM_DELAY)

    # ── Hybrid System ──
    print(f"  [HYBRID] Running...")
    hyb_result = hybrid_engine.query(q["question"], mode=q["mode"])
    print(f"  [HYBRID] Done: case={hyb_result.retrieval_case} "
          f"source={hyb_result.knowledge_source} "
          f"conf={hyb_result.confidence} "
          f"docs={len(hyb_result.selected_docs)} "
          f"total={hyb_result.total_time:.2f}s")
    if hyb_result.error:
        print(f"  [HYBRID] ERROR: {hyb_result.error}")

    # ── Store all three ──
    comparison_results.append({
        "query_id": q["id"],
        "question": q["question"],
        "mode": q["mode"],
        "chapter": q["chapter"],
        "difficulty": q["difficulty"],
        # RAG
        "rag_answer": rag_result.answer or "",
        "rag_error": rag_result.error,
        "rag_case": rag_result.retrieval_case,
        "rag_confidence": rag_result.confidence,
        "rag_n_docs": len(rag_result.selected_docs),
        "rag_best_dist": round(rag_result.selected_docs[0].distance, 4) if rag_result.selected_docs else None,
        "rag_retrieval_time": round(rag_result.retrieval_time, 3),
        "rag_generation_time": round(rag_result.generation_time, 3),
        "rag_total_time": round(rag_result.total_time, 3),
        "rag_answer_length": len(rag_result.answer or ""),
        # Prompt-Only
        "po_answer": po_result.answer or "",
        "po_error": po_result.error,
        "po_confidence": po_result.confidence,
        "po_generation_time": round(po_result.generation_time, 3),
        "po_total_time": round(po_result.total_time, 3),
        "po_answer_length": len(po_result.answer or ""),
        "po_sys_tokens": po_result.system_prompt_tokens_approx,
        # Hybrid
        "hyb_answer": hyb_result.answer or "",
        "hyb_error": hyb_result.error,
        "hyb_case": hyb_result.retrieval_case,
        "hyb_knowledge_source": hyb_result.knowledge_source,
        "hyb_confidence": hyb_result.confidence,
        "hyb_n_docs": len(hyb_result.selected_docs),
        "hyb_best_dist": round(hyb_result.best_distance, 4) if hyb_result.best_distance else None,
        "hyb_retrieval_time": round(hyb_result.retrieval_time, 3),
        "hyb_generation_time": round(hyb_result.generation_time, 3),
        "hyb_total_time": round(hyb_result.total_time, 3),
        "hyb_answer_length": len(hyb_result.answer or ""),
        "hyb_sys_tokens": hyb_result.system_prompt_tokens_approx,
    })

    print(f"\n  Answer lengths: RAG={len(rag_result.answer or '')} | "
          f"PO={len(po_result.answer or '')} | "
          f"Hybrid={len(hyb_result.answer or '')}")

    if i < len(EVAL_QUERIES):
        print(f"\n  [DELAY] Waiting {INTER_QUERY_DELAY}s before next query...")
        time.sleep(INTER_QUERY_DELAY)

print(f"\n\n{'='*80}")
print(f"Three-way evaluation complete: {len(comparison_results)} queries.")

---
## 3. Hybrid Routing Analysis

Key thesis metric: how often does the router pick each case?

In [ ]:
from collections import Counter

case_dist = Counter(r['hyb_case'] for r in comparison_results)
source_dist = Counter(r['hyb_knowledge_source'] for r in comparison_results)

print("=== Hybrid Routing Distribution ===")
print(f"\n  Case distribution:")
for case in ["A", "B", "C"]:
    count = case_dist.get(case, 0)
    pct = 100 * count / len(comparison_results)
    bar = '#' * int(pct / 2)
    label = {"A": "Strong retrieval", "B": "Weak retrieval (hybrid)", "C": "Parametric fallback"}[case]
    print(f"    Case {case} ({label:30s}): {count:2d}/{len(comparison_results)} ({pct:5.1f}%)  {bar}")

print(f"\n  Knowledge source distribution:")
for src, count in source_dist.most_common():
    pct = 100 * count / len(comparison_results)
    print(f"    {src:15s}: {count:2d}/{len(comparison_results)} ({pct:5.1f}%)")

print(f"\n  Per-query routing:")
for r in comparison_results:
    bd = f"{r['hyb_best_dist']:.4f}" if r['hyb_best_dist'] else '  N/A '
    print(f"    {r['query_id']} | {r['chapter']:<25s} | Case {r['hyb_case']} | "
          f"source={r['hyb_knowledge_source']:12s} | best_dist={bd}")

---
## 4. Three-Way Quantitative Comparison

In [ ]:
hdr = (f"{'ID':>4s}  {'Chapter':<22s}  {'Mode':<8s}  "
       f"{'RAG_T':>6s} {'PO_T':>6s} {'HYB_T':>6s}  "
       f"{'RAG_L':>6s} {'PO_L':>6s} {'HYB_L':>6s}  "
       f"{'R_Cf':>5s} {'P_Cf':>5s} {'H_Cf':>5s}  "
       f"{'H_Case':>6s}")
sep = '-' * len(hdr)

print(sep)
print(hdr)
print(sep)

for r in comparison_results:
    ch = r['chapter'][:20] + '..' if len(r['chapter']) > 22 else r['chapter']
    print(f"{r['query_id']:>4s}  {ch:<22s}  {r['mode']:<8s}  "
          f"{r['rag_total_time']:6.2f} {r['po_total_time']:6.2f} {r['hyb_total_time']:6.2f}  "
          f"{r['rag_answer_length']:6d} {r['po_answer_length']:6d} {r['hyb_answer_length']:6d}  "
          f"{r['rag_confidence']:>5s} {r['po_confidence']:>5s} {r['hyb_confidence']:>5s}  "
          f"     {r['hyb_case']}")

print(sep)

# ── Aggregate statistics ──
rag_times = [r['rag_total_time'] for r in comparison_results]
po_times = [r['po_total_time'] for r in comparison_results]
hyb_times = [r['hyb_total_time'] for r in comparison_results]
rag_lens = [r['rag_answer_length'] for r in comparison_results]
po_lens = [r['po_answer_length'] for r in comparison_results]
hyb_lens = [r['hyb_answer_length'] for r in comparison_results]
rag_ok = sum(1 for r in comparison_results if r['rag_answer'])
po_ok = sum(1 for r in comparison_results if r['po_answer'])
hyb_ok = sum(1 for r in comparison_results if r['hyb_answer'])

n = len(comparison_results)
print(f"\n  AGGREGATE STATISTICS")
print(f"  {'Metric':<30s}  {'RAG':>10s}  {'Prompt-Only':>12s}  {'Hybrid':>10s}")
print(f"  {'─'*68}")
print(f"  {'Success rate':<30s}  {rag_ok:>8d}/10  {po_ok:>10d}/10  {hyb_ok:>8d}/10")
print(f"  {'Avg total time (s)':<30s}  {sum(rag_times)/n:>10.2f}  {sum(po_times)/n:>12.2f}  {sum(hyb_times)/n:>10.2f}")
print(f"  {'Avg answer length (chars)':<30s}  {sum(rag_lens)/n:>10.0f}  {sum(po_lens)/n:>12.0f}  {sum(hyb_lens)/n:>10.0f}")
print(f"  {'Total API calls':<30s}  {'10':>10s}  {'10':>12s}  {'10':>10s}")
print(f"  {'Embedding calls':<30s}  {'10':>10s}  {'0':>12s}  {'10':>10s}")

---
## 5. Side-by-Side Answer Inspection

Change `QUERY_INDEX` to view different queries.

In [ ]:
QUERY_INDEX = 0  # Change this to view different queries (0-9)

r = comparison_results[QUERY_INDEX]

print(f"{'='*80}")
print(f"  {r['query_id']}: {r['chapter']} ({r['mode']})")
print(f"  Q: {r['question']}")
print(f"{'='*80}")

print(f"\n{'─'*40}")
print(f"  RAG")
print(f"  Case={r['rag_case']} | Conf={r['rag_confidence']} | "
      f"Docs={r['rag_n_docs']} | Time={r['rag_total_time']}s")
print(f"{'─'*40}")
print(r['rag_answer'][:1500] if r['rag_answer'] else f"ERROR: {r['rag_error']}")
if len(r['rag_answer']) > 1500:
    print(f"\n[...truncated, {len(r['rag_answer'])} total chars]")

print(f"\n\n{'─'*40}")
print(f"  PROMPT-ONLY")
print(f"  Conf={r['po_confidence']} | Time={r['po_total_time']}s")
print(f"{'─'*40}")
print(r['po_answer'][:1500] if r['po_answer'] else f"ERROR: {r['po_error']}")
if len(r['po_answer']) > 1500:
    print(f"\n[...truncated, {len(r['po_answer'])} total chars]")

print(f"\n\n{'─'*40}")
print(f"  HYBRID")
print(f"  Case={r['hyb_case']} | Source={r['hyb_knowledge_source']} | "
      f"Conf={r['hyb_confidence']} | Docs={r['hyb_n_docs']} | Time={r['hyb_total_time']}s")
print(f"{'─'*40}")
print(r['hyb_answer'][:1500] if r['hyb_answer'] else f"ERROR: {r['hyb_error']}")
if len(r['hyb_answer']) > 1500:
    print(f"\n[...truncated, {len(r['hyb_answer'])} total chars]")

---
## 6. Syllabus Guard — Three-Way

All three systems should refuse out-of-program methods.

In [ ]:
GUARD_QUERIES = [
    "Utilise la règle de L'Hôpital pour calculer lim(x→0) sin(x)/x",
    "Diagonalise la matrice A = [[2,1],[0,3]] pour calculer A^n",
]

for idx, gq in enumerate(GUARD_QUERIES):
    print(f"\n{'='*80}")
    print(f"  SYLLABUS GUARD: {gq}")
    print(f"{'='*80}")

    for name, eng in [("RAG", rag_engine), ("PROMPT-ONLY", po_engine), ("HYBRID", hybrid_engine)]:
        result = eng.query(gq, mode="correction")
        answer = result.answer or f"ERROR: {result.error}"
        print(f"\n  [{name}] (first 400 chars):")
        print(f"  {'─'*40}")
        for line in answer[:400].split('\n'):
            print(f"  {line}")
        if len(answer) > 400:
            print(f"  [...truncated, {len(answer)} total chars]")
        time.sleep(2)

    if idx < len(GUARD_QUERIES) - 1:
        time.sleep(5)

---
## 7. Export for Three-Way Blind Grading

In [ ]:
import random

random.seed(42)

blind_sheets = []
answer_key = []

system_labels = ["RAG", "PROMPT_ONLY", "HYBRID"]

for r in comparison_results:
    # Randomly assign the three systems to System A / B / C
    answers = {
        "RAG": r['rag_answer'],
        "PROMPT_ONLY": r['po_answer'],
        "HYBRID": r['hyb_answer'],
    }
    shuffled = list(system_labels)
    random.shuffle(shuffled)

    blind_sheets.append({
        "query_id": r['query_id'],
        "question": r['question'],
        "mode": r['mode'],
        "chapter": r['chapter'],
        "system_a_answer": answers[shuffled[0]],
        "system_b_answer": answers[shuffled[1]],
        "system_c_answer": answers[shuffled[2]],
        "grading": {
            lbl: {
                "mathematical_correctness": None,
                "formal_rigor": None,
                "official_style_alignment": None,
                "clarity_and_structure": None,
                "pedagogical_quality": None,
                "hallucination_detected": None,
                "overall_score": None,
                "comments": "",
            }
            for lbl in ["system_a", "system_b", "system_c"]
        },
    })

    answer_key.append({
        "query_id": r['query_id'],
        "system_a_is": shuffled[0],
        "system_b_is": shuffled[1],
        "system_c_is": shuffled[2],
    })

# ── Save files ──
with open("blind_grading_sheets_3way.json", "w", encoding="utf-8") as f:
    json.dump(blind_sheets, f, ensure_ascii=False, indent=2)
print(f"Blind grading sheets saved: blind_grading_sheets_3way.json")

with open("answer_key_3way.json", "w", encoding="utf-8") as f:
    json.dump(answer_key, f, ensure_ascii=False, indent=2)
print(f"Answer key saved: answer_key_3way.json (keep secret!)")

full_export = {
    "metadata": {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "model": config.CHAT_MODEL_ID,
        "rag_chunks": rag_engine.chunk_count,
        "temperature": 0.15,
        "max_tokens": 4096,
        "thresholds": {
            "good": config.SIMILARITY_GOOD_THRESHOLD,
            "fallback": config.SIMILARITY_FALLBACK_THRESHOLD,
        },
    },
    "queries": comparison_results,
    "answer_key": answer_key,
}

with open("comparison_three_way_results.json", "w", encoding="utf-8") as f:
    json.dump(full_export, f, ensure_ascii=False, indent=2)
print(f"Full results saved: comparison_three_way_results.json")

print(f"\nAnswer key (for your eyes only):")
for ak in answer_key:
    print(f"  {ak['query_id']}: A={ak['system_a_is']}, B={ak['system_b_is']}, C={ak['system_c_is']}")

---
## 8. Thesis-Ready Summary Report

In [ ]:
rag_successes = sum(1 for r in comparison_results if r['rag_answer'])
po_successes = sum(1 for r in comparison_results if r['po_answer'])
hyb_successes = sum(1 for r in comparison_results if r['hyb_answer'])
n = len(comparison_results)
avg_rag_time = sum(r['rag_total_time'] for r in comparison_results) / n
avg_po_time = sum(r['po_total_time'] for r in comparison_results) / n
avg_hyb_time = sum(r['hyb_total_time'] for r in comparison_results) / n
avg_rag_len = sum(r['rag_answer_length'] for r in comparison_results) / n
avg_po_len = sum(r['po_answer_length'] for r in comparison_results) / n
avg_hyb_len = sum(r['hyb_answer_length'] for r in comparison_results) / n

case_a_count = sum(1 for r in comparison_results if r['hyb_case'] == 'A')
case_b_count = sum(1 for r in comparison_results if r['hyb_case'] == 'B')
case_c_count = sum(1 for r in comparison_results if r['hyb_case'] == 'C')

print(f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║       RAG vs PROMPT-ONLY vs HYBRID — THREE-WAY EVALUATION REPORT          ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  Experimental Setup                                                        ║
║    LLM Model        : {config.CHAT_MODEL_ID:<52s} ║
║    Temperature       : 0.15                                                ║
║    Max tokens         : 4096                                               ║
║    Eval queries       : {n:<51d} ║
║                                                                            ║
║  ┌────────────────────────────────────────────────────────────────────────┐ ║
║  │ Metric                    RAG        Prompt-Only     Hybrid           │ ║
║  │ ─────────────────────────────────────────────────────────────────── │ ║
║  │ Success rate          {rag_successes:>5d}/10      {po_successes:>5d}/10       {hyb_successes:>5d}/10          │ ║
║  │ Avg total time (s)   {avg_rag_time:>8.2f}      {avg_po_time:>8.2f}       {avg_hyb_time:>8.2f}          │ ║
║  │ Avg answer length    {avg_rag_len:>8.0f}      {avg_po_len:>8.0f}       {avg_hyb_len:>8.0f}          │ ║
║  └────────────────────────────────────────────────────────────────────────┘ ║
║                                                                            ║
║  Hybrid Routing Distribution                                               ║
║    Case A (strong retrieval)    : {case_a_count:>2d}/{n}                                    ║
║    Case B (weak → hybrid)       : {case_b_count:>2d}/{n}                                    ║
║    Case C (parametric fallback) : {case_c_count:>2d}/{n}                                    ║
║                                                                            ║
║  Exports                                                                   ║
║    blind_grading_sheets_3way.json    (for teacher)                         ║
║    answer_key_3way.json              (reveal after grading)                ║
║    comparison_three_way_results.json (full data for thesis)                ║
║                                                                            ║
╚══════════════════════════════════════════════════════════════════════════════╝
""")

---
## 9. Analysis: Hybrid System Insights

### Expected Hybrid Advantages:
- **Best of both worlds:** Uses RAG when retrieval is strong, parametric knowledge when it's not
- **No wasted retrieval:** Doesn't force bad retrieval into the prompt (unlike pure RAG)
- **Curriculum safety net:** Case B merges partial retrieval with curriculum knowledge
- **Transparent routing:** `knowledge_source` field shows WHY each path was chosen

### Key Thesis Questions:
1. **Does Case B outperform both RAG and Prompt-Only** for borderline queries?
2. **Does Case C match Prompt-Only quality** (it should — same prompt)?
3. **Does Case A match RAG quality** (it should — same pipeline)?
4. **What is the routing distribution** on realistic student queries?
5. **Is the overhead of always-attempting-retrieval justified** by the quality improvement?

### Grading Rubric (5 criteria × 3 systems × 10 queries):
| Criterion | Weight | Description |
|---|---|---|
| Mathematical correctness | 5/20 | Are calculations and proofs correct? |
| Formal rigor | 5/20 | Formal mathematical writing conventions? |
| Official style alignment | 3/20 | Matches Tunisian Bac correction style? |
| Clarity & structure | 3/20 | Well-organized and easy to follow? |
| Pedagogical quality | 2/20 | Teaches effectively (coaching mode)? |
| Hallucination check | 2/20 | Any fabricated facts or wrong theorems? |